# 03 — Lightweight Classification: HOG/SVM → MobileNet

Pipeline Sentinel learning notebook. Restart-safe comparison of hand-engineered HOG features, a linear SVM, and pretrained MobileNetV3 features.


## Learning objectives

Create labeled crops, train a cheap HOG/SVM baseline, understand proposal → classifier separation, inspect raw MobileNet ImageNet predictions, and compare frozen MobileNet features with the same linear-classifier idea.


In [ ]:
from pathlib import Path
import sys
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)

print("Python:", sys.executable)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the Pipeline Sentinel project root from either the root or notebooks/ directory."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find the Pipeline Sentinel project root. "
        "Open this notebook from inside the course folder containing data/, notebooks/, and src/."
    )

ROOT = find_project_root()
RAW = ROOT / "data" / "raw"
VIDEO = RAW / "pipeline_demo_eo.mp4"
GT_CSV = RAW / "pipeline_demo_eo_gt.csv"

print("Project root:", ROOT)
print("Video:", VIDEO)
print("Ground truth:", GT_CSV)


In [ ]:
# Make the notebook recover gracefully if the demo files were deleted.
if not VIDEO.exists() or not GT_CSV.exists():
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    try:
        from src.synthetic import generate_demo_video
        print("Demo files missing — regenerating deterministic EO sequence...")
        generate_demo_video(VIDEO, GT_CSV, modality="EO", seed=7)
    except Exception as exc:
        raise FileNotFoundError(
            f"Required demo files are missing and could not be regenerated:\n"
            f"  {VIDEO}\n  {GT_CSV}\n"
            "Keep the notebook inside the complete Pipeline Sentinel course folder."
        ) from exc

assert VIDEO.exists(), VIDEO
assert GT_CSV.exists(), GT_CSV
print("Demo inputs ready.")


In [ ]:
gt = pd.read_csv(GT_CSV)

required = {
    "frame_number", "timestamp_s", "label", "object_id",
    "x1", "y1", "x2", "y2", "scenario_role"
}
missing = required - set(gt.columns)
if missing:
    raise ValueError(f"Ground-truth file is missing columns: {sorted(missing)}")

print("Annotation rows:", len(gt))
print("Labels:")
print(gt["label"].value_counts())

display(gt.head())


In [ ]:
def read_all_frames(video_path: Path) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {video_path}")

    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)

    cap.release()

    if not frames:
        raise RuntimeError(f"Video opened but yielded zero frames: {video_path}")

    return frames

start = time.perf_counter()
frames = read_all_frames(VIDEO)
elapsed = time.perf_counter() - start

print(f"Decoded {len(frames)} frames in {elapsed:.3f} s")
print("Frame shape:", frames[0].shape)


In [ ]:
# Validate that every annotation references a frame that actually exists.
max_annotated_frame = int(gt["frame_number"].max())
if max_annotated_frame >= len(frames):
    raise IndexError(
        f"GT references frame {max_annotated_frame}, but video contains only {len(frames)} frames."
    )

print("GT/video frame references are consistent.")


In [ ]:
def crop_xyxy(frame: np.ndarray, box, pad: int = 0) -> np.ndarray:
    """Safe XYXY crop clipped to the frame bounds."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)

    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)

    if x2 <= x1 or y2 <= y1:
        return np.empty((0, 0, 3), dtype=frame.dtype)

    return frame[y1:y2, x1:x2].copy()


In [ ]:
object_gt = gt.loc[gt["label"] != "smoke"].copy().reset_index(drop=True)

images = []
labels = []
crop_meta = []

for row in object_gt.itertuples(index=False):
    frame = frames[int(row.frame_number)]
    crop = crop_xyxy(
        frame,
        (row.x1, row.y1, row.x2, row.y2),
        pad=3,
    )

    if crop.size == 0:
        continue

    images.append(crop)
    labels.append(str(row.label))
    crop_meta.append({
        "frame_number": int(row.frame_number),
        "object_id": int(row.object_id),
        "label": str(row.label),
        "scenario_role": str(row.scenario_role),
    })

if not images:
    raise RuntimeError("No valid object crops were produced.")

crop_meta = pd.DataFrame(crop_meta)

print("Crops:", len(images))
print("Class counts:")
print(pd.Series(labels).value_counts())


In [ ]:
def hog_features(img: np.ndarray) -> np.ndarray:
    """Convert one BGR object crop into a 1-D HOG feature vector."""
    resized = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )

    return features.astype(np.float32)

example_features = hog_features(images[0])
print("One 64×64 crop becomes", example_features.shape[0], "HOG features")
print("Feature dtype:", example_features.dtype)


In [ ]:
start = time.perf_counter()
X = np.vstack([hog_features(img) for img in images])
y = np.asarray(labels)
elapsed = time.perf_counter() - start

print("Feature matrix X:", X.shape)
print("Label vector y:", y.shape)
print(f"HOG extraction time: {elapsed:.3f} s")
print("Classes:", sorted(np.unique(y)))


In [ ]:
class_counts = pd.Series(y).value_counts()
if class_counts.min() < 2:
    raise ValueError(
        "At least two samples per class are required for a stratified split. "
        f"Counts: {class_counts.to_dict()}"
    )

sample_indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    sample_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

Xtr, Xte = X[train_idx], X[test_idx]
ytr, yte = y[train_idx], y[test_idx]

print("Train samples:", len(train_idx))
print("Test samples:", len(test_idx))
print("Train classes:", pd.Series(ytr).value_counts().to_dict())
print("Test classes:", pd.Series(yte).value_counts().to_dict())


In [ ]:
clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=0.5, random_state=SEED),
)

start = time.perf_counter()
clf.fit(Xtr, ytr)
train_seconds = time.perf_counter() - start

yp = clf.predict(Xte)

print(f"Training time: {train_seconds:.4f} s")
print()
print(classification_report(yte, yp, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    yte,
    yp,
    xticks_rotation=45,
    cmap="Blues",
    colorbar=False,
)
plt.title("HOG + Linear SVM — synthetic demonstration")
plt.tight_layout()
plt.show()


## Proposal → classifier architecture

The classifier answers **what** is in a crop; a proposal mechanism decides **where** to crop. Keeping those responsibilities separate is an important edge-pipeline pattern.


In [ ]:
def classify_proposals(
    frame: np.ndarray,
    proposal_boxes,
    classifier,
) -> list[dict]:
    """Classify candidate XYXY boxes using the trained HOG/SVM pipeline."""
    results = []

    for box in proposal_boxes:
        crop = crop_xyxy(frame, box, pad=3)
        if crop.size == 0:
            continue

        features = hog_features(crop).reshape(1, -1)
        label = classifier.predict(features)[0]

        results.append({
            "box": tuple(map(int, box)),
            "label": str(label),
        })

    return results

# Smoke-test the interface with several known GT boxes from one frame.
demo_frame_number = 100
demo_rows = object_gt.loc[object_gt["frame_number"] == demo_frame_number]
demo_boxes = [tuple(r) for r in demo_rows[["x1", "y1", "x2", "y2"]].to_numpy()]

demo_results = classify_proposals(
    frames[demo_frame_number],
    demo_boxes,
    clf,
)

demo_results


## MobileNetV3 comparison

Two tests are intentionally separated: raw ImageNet top-5 predictions show what the untouched pretrained network calls our crops; frozen MobileNet features plus a linear probe test whether its learned representation is useful for our mission labels.


In [ ]:
RUN_MOBILENET = True

mobilenet_ready = False

if RUN_MOBILENET:
    try:
        import torch
        import torch.nn as nn
        from PIL import Image
        from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

        weights = MobileNet_V3_Small_Weights.DEFAULT
        model = mobilenet_v3_small(weights=weights).eval()
        preprocess = weights.transforms()
        imagenet_categories = weights.meta['categories']

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)
        mobilenet_ready = True

        print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
        print('Device:', device)
        print('ImageNet categories:', len(imagenet_categories))
        print('MobileNetV3-Small is ready.')
    except Exception as exc:
        print('MobileNet could not be initialized:')
        print(type(exc).__name__ + ':', exc)
else:
    print('MobileNet skipped. Set RUN_MOBILENET=True when ready.')


### Raw pretrained ImageNet assessment

This is diagnostic rather than a mission classifier because ImageNet's vocabulary is not `vehicle/person/aerial_object`.


In [ ]:
if mobilenet_ready:
    # Deterministic sample: three crops from each mission class.
    sample_indices = []
    for class_name in sorted(pd.Series(labels).unique()):
        class_idx = np.where(np.asarray(labels) == class_name)[0][:3]
        sample_indices.extend(class_idx.tolist())

    rows = []
    fig, axes = plt.subplots(len(sample_indices), 1, figsize=(10, 3.2 * len(sample_indices)))
    if len(sample_indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, sample_indices):
        crop_bgr = images[idx]
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        pil = Image.fromarray(crop_rgb)
        tensor = preprocess(pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1)[0]
            top_probs, top_ids = probs.topk(5)

        top_labels = [imagenet_categories[int(i)] for i in top_ids.cpu()]
        top_probs_np = top_probs.cpu().numpy()
        meta = crop_meta.iloc[idx]

        row = {
            'crop_index': int(idx),
            'mission_truth': labels[idx],
            'frame_number': int(meta.frame_number),
            'object_id': int(meta.object_id),
        }
        for rank, (name, prob) in enumerate(zip(top_labels, top_probs_np), start=1):
            row[f'top{rank}'] = name
            row[f'p{rank}'] = float(prob)
        rows.append(row)

        ax.imshow(crop_rgb)
        ax.set_title(
            f"MISSION TRUTH: {labels[idx]} | MobileNet top-1: {top_labels[0]} ({top_probs_np[0]:.1%})\n"
            + ' | '.join(f'{name}: {prob:.1%}' for name, prob in zip(top_labels[:3], top_probs_np[:3]))
        )
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    mobilenet_raw_predictions = pd.DataFrame(rows)
    display_cols = ['crop_index', 'mission_truth', 'frame_number', 'object_id',
                    'top1', 'p1', 'top2', 'p2', 'top3', 'p3', 'top4', 'p4', 'top5', 'p5']
    display(mobilenet_raw_predictions[display_cols])
else:
    print('MobileNet is not ready, so raw predictions were skipped.')


### Frozen MobileNet features + linear probe

Remove the ImageNet classifier, extract the learned feature vector, and train a linear SVM on the same train/test split used by HOG.


In [ ]:
if mobilenet_ready:
    feature_model = nn.Sequential(
        model.features,
        model.avgpool,
        nn.Flatten(1),
    ).to(device).eval()

    def mobilenet_embeddings(image_list, batch_size=32):
        chunks = []
        for start_idx in range(0, len(image_list), batch_size):
            batch_imgs = image_list[start_idx:start_idx + batch_size]
            tensors = []
            for img in batch_imgs:
                rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                tensors.append(preprocess(Image.fromarray(rgb)))
            batch = torch.stack(tensors).to(device)
            with torch.no_grad():
                z = feature_model(batch)
            chunks.append(z.cpu().numpy())
        return np.vstack(chunks)

    start = time.perf_counter()
    X_mobile = mobilenet_embeddings(images)
    embed_seconds = time.perf_counter() - start

    print('MobileNet feature matrix:', X_mobile.shape)
    print(f'Embedding time: {embed_seconds:.3f} s')
    print(f'Average: {1000 * embed_seconds / len(images):.2f} ms/crop')

    Xtr_mobile = X_mobile[train_idx]
    Xte_mobile = X_mobile[test_idx]

    mobile_clf = make_pipeline(
        StandardScaler(),
        LinearSVC(C=0.5, random_state=SEED),
    )

    start = time.perf_counter()
    mobile_clf.fit(Xtr_mobile, ytr)
    mobile_train_seconds = time.perf_counter() - start
    yp_mobile = mobile_clf.predict(Xte_mobile)

    print(f'Linear-probe training time: {mobile_train_seconds:.4f} s')
    print()
    print(classification_report(yte, yp_mobile, zero_division=0))
else:
    print('MobileNet is not ready, so the frozen-feature probe was skipped.')


In [ ]:
if mobilenet_ready:
    print('MobileNet confusion matrix')
    ConfusionMatrixDisplay.from_predictions(
        yte,
        yp_mobile,
        xticks_rotation=45,
        cmap='Blues',
        colorbar=False,
    )
    plt.title('Frozen MobileNetV3 features + Linear SVM')
    plt.tight_layout()
    plt.show()

    hog_accuracy = float(np.mean(yp == yte))
    mobile_accuracy = float(np.mean(yp_mobile == yte))

    comparison = pd.DataFrame([
        {
            'representation': 'HOG',
            'feature_dimensions': int(X.shape[1]),
            'test_accuracy': hog_accuracy,
            'feature_type': 'hand-engineered gradients',
        },
        {
            'representation': 'MobileNetV3-Small',
            'feature_dimensions': int(X_mobile.shape[1]),
            'test_accuracy': mobile_accuracy,
            'feature_type': 'pretrained learned CNN features',
        },
    ])

    display(comparison)


## What we learned

A cheap HOG/SVM baseline provides a control. MobileNet lets us distinguish pretrained class vocabulary from representation quality. The next production step is a detector adapter that localizes and classifies through a stable Pipeline Sentinel `Detection` contract.
